# 🎯 Object Detection — Colab'da eğitim

Bu notebook, projedeki `scripts/train.py` ile **aynı** eğitimi Colab'ın ücretsiz
GPU'sunda çalıştırır. Mac'te ~45 sn/epoch süren eğitim burada birkaç saniyeye iner.

**Kullanım:** `Runtime → Change runtime type → T4 GPU` seç, sonra hücreleri sırayla çalıştır.

Sonunda eğitilen `best.pt` dosyasını indirip projenin `models/` klasörüne koy —
arayüz onu otomatik olarak *"Ozel: ..."* diye model listesine ekler.


## 1. GPU kontrolü


In [ ]:
!nvidia-smi


## 2. Kurulum

`lap` ByteTrack için gerekli; takip kodunu burada kullanmasak da
projenin bağımlılıklarıyla aynı kalsın diye kuruyoruz.


In [ ]:
!pip install -q ultralytics 'lap>=0.5.12'

import torch, ultralytics
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
print('ultralytics:', ultralytics.__version__)


## 3. Ayarlar

Kendi veri setinle eğitmek istersen `DATA`'yı kendi `data.yaml` yoluna çevir.


In [ ]:
DATA   = 'african-wildlife.yaml'   # ultralytics hazır seti; veya kendi data.yaml'in
MODEL  = 'yolov8n.pt'               # başlangıç ağırlığı
EPOCHS = 60                         # GPU'da bol tutabiliriz
IMGSZ  = 640
BATCH  = 32
NAME   = DATA.replace('.yaml', '')


## 4. Eğitim

`patience=15`: 15 epoch boyunca iyileşme olmazsa erken durur, boşuna GPU yakmayız.


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=15,
    name=NAME,
    plots=True,
)
print('çıktılar:', results.save_dir)


## 5. Doğrulama metrikleri

Bu tabloyu README'deki metrik tablosuyla karşılaştırabilirsin.


In [ ]:
metrics = model.val(data=DATA)

print(f'mAP50    : {metrics.box.map50:.3f}')
print(f'mAP50-95 : {metrics.box.map:.3f}')
print(f'precision: {metrics.box.mp:.3f}')
print(f'recall   : {metrics.box.mr:.3f}')
print()
for i, c in enumerate(metrics.box.ap_class_index):
    print(f'{model.names[int(c)]:12} mAP50={metrics.box.ap50[i]:.3f}')


## 6. Eğitim grafikleri


In [ ]:
from IPython.display import Image, display
from pathlib import Path

run = Path(results.save_dir)
for plot in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png']:
    path = run / plot
    if path.exists():
        print(plot)
        display(Image(filename=str(path), width=800))


## 7. Modeli indir

İnen dosyayı projenin `models/` klasörüne `african-wildlife.pt` adıyla koy.


In [ ]:
from google.colab import files
from pathlib import Path

best = Path(results.save_dir) / 'weights' / 'best.pt'
target = Path(f'/content/{NAME}.pt')
target.write_bytes(best.read_bytes())
print('boyut:', round(target.stat().st_size / 1e6, 1), 'MB')
files.download(str(target))
